# Data exploration

Exploration of data structure and edge cases

## Config

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd

DATA_DIR = Path("..") / "data" / "raw"
EXPORT_DATE = pd.Timestamp("2026-10-01")

## Helper functions

In [2]:
def load_raw_csv(path: Path, encoding: str, separator: str) -> pd.DataFrame:
    """Read a CSV as raw strings, keeping blanks as empty strings.

    Args:
        path: Path to the CSV file.
        encoding: File encoding, detected from the raw bytes.
        separator: Field separator.

    Returns:
        A DataFrame where every column has string dtype and no value is
        turned into NaN.
    """
    return pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        encoding=encoding,
        sep=separator,
    )

In [3]:
def value_shapes(values: pd.Series) -> pd.Series:
    """Summarise string formats by masking their characters.

    Digits become 9 and letters become a or A, so distinct formats (for
    example "9999-99-99" vs "9. 9. 9999") collapse into countable patterns.

    Args:
        values: A string column.

    Returns:
        Counts of each masked pattern, most frequent first.
    """
    masked = (
        values.str.replace(r"\d", "9", regex=True)
        .str.replace(r"[a-záčďéěíňóřšťúůýž]", "a", regex=True)
        .str.replace(r"[A-ZÁČĎÉĚÍŇÓŘŠŤÚŮÝŽ]", "A", regex=True)
    )
    return masked.value_counts()

In [4]:
def profile_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Describe each column by blanks, distinct values and a first example.

    Args:
        frame: Any raw string DataFrame.

    Returns:
        One row per column with the count of empty strings, the number of
        unique values and the first non-empty value seen.
    """
    return pd.DataFrame(
        {
            "empty": (frame == "").sum(),
            "unique": frame.nunique(),
            "example": frame.apply(lambda col: col[col != ""].iloc[0] if (col != "").any() else ""),
        }
    )

In [5]:
def remove_diacritics(text: str) -> str:
    """Remove diacritics so that spellings with and without them compare equal.

    Args:
        text: Any text.

    Returns:
        The text without diacritics, e.g. 'Správa budov' -> 'Sprava budov'.
    """
    # NFKD splits a letter like "á" into "a" plus a separate accent mark.
    split_letters = unicodedata.normalize("NFKD", text)
    return "".join(char for char in split_letters if not unicodedata.combining(char))

In [6]:
def parse_mixed_dates(values: pd.Series) -> pd.Series:
    """Parse ISO and Czech dates (2019-01-06, 6. 1. 2019, 06.01.2019) for exploration.

    Args:
        values: Raw date strings.

    Returns:
        Parsed dates. Excel serial numbers and invalid values become NaT.
    """
    compact = values.str.replace(" ", "")
    iso_dates = pd.to_datetime(compact, format="%Y-%m-%d", errors="coerce")
    czech_dates = pd.to_datetime(compact, format="%d.%m.%Y", errors="coerce")
    return iso_dates.fillna(czech_dates)

## 1. Loading

Encodings and separators were detected from the raw bytes

In [7]:
employees_df = load_raw_csv(DATA_DIR / "zamestnanci.csv", encoding="utf-8-sig", separator=",")
completed_df = load_raw_csv(DATA_DIR / "absolvovana_skoleni.csv", encoding="cp1250", separator=";")
catalog_df = load_raw_csv(DATA_DIR / "katalog_skoleni.csv", encoding="utf-8-sig", separator=",")
dates_df = load_raw_csv(DATA_DIR / "terminy_skoleni.csv", encoding="utf-8-sig", separator=";")

In [8]:
raw_frames = {
    "employees": employees_df,
    "completed": completed_df,
    "catalog": catalog_df,
    "dates": dates_df,
}
for name, frame in raw_frames.items():
    rows, columns = frame.shape
    print(f"{name:<10}{rows:>5} rows {columns:>3} columns")
    print(f"{'':<10}{', '.join(frame.columns)}")
    print()

employees   174 rows  12 columns
          id_zamestnance, jmeno, prijmeni, email, utvar, pozice, vedouci_email, datum_nastupu, datum_ukonceni, stav, rodne_cislo, poznamka

completed  2868 rows   9 columns
          id_zaznamu, id_zamestnance, zamestnanec, kod_skoleni, datum_absolvovani, platnost_do, poskytovatel, cena_kc, poznamka

catalog      10 rows   5 columns
          kod, nazev, platnost_mesicu, povinne_pro, poznamka

dates        34 rows   9 columns
          id_terminu, kod_skoleni, datum, cas, misto, kapacita, obsazeno, cena_na_osobu, stav



## 2. Catalog (`katalog_skoleni.csv`)


In [9]:
catalog_df

,kod,nazev,platnost_mesicu,povinne_pro,poznamka
0,BOZP-Z,BOZP – základní školení,24,VSICHNI,
1,POZ-OCH,Požární ochrana,24,VSICHNI,
2,GDPR,Ochrana osobních údajů,36,VSICHNI,
3,KYBER,Kybernetická bezpečnost,12,VSICHNI,
4,PRV-POM,První pomoc,36,UTVAR:Laboratoře;Správa budov;Knihovna,
5,RIDIC,Řidiči referenti,12,POZICE:Řidič,
6,LAB-BEZ,Bezpečná práce v laboratoři,24,UTVAR:Laboratoře,
7,ZDVIH,Obsluha zdvihacích zařízení,2 roky,POZICE:Technik údržby,
8,LEAD,Základy vedení lidí,0,POZICE:Vedoucí,
9,ANGL,Anglický jazyk – kurz,,NEPOVINNE,


## 3. Employees (`zamestnanci.csv`)

`rodne_cislo` - personal data, therefore dropped immediately after loading.

In [10]:
print(f"rodne_cislo non-empty: {(employees_df['rodne_cislo'] != '').sum()} of {len(employees_df)}")
employees_df = employees_df.drop(columns="rodne_cislo")

rodne_cislo non-empty: 174 of 174


In [11]:
contact_columns = ["email", "vedouci_email"]
profile_columns(employees_df.drop(columns=contact_columns))

,empty,unique,example
id_zamestnance,0,174,E0001
jmeno,0,65,Tereza
prijmeni,0,103,Urbanová
utvar,5,18,Studijní oddělení
pozice,0,11,Vedoucí
datum_nastupu,0,166,2008-02-23
datum_ukonceni,158,17,2027-01-20
stav,0,3,aktivní
poznamka,170,5,-2+3


Detecting unexpected or inconsistent category values: unknown departments or positions, casing and spelling variants, and whether `stav` is drawn from a small fixed set.

In [12]:
for column in ["stav", "utvar", "pozice"]:
    print(employees_df[column].value_counts(dropna=False))
    print()

stav
aktivní       156
ukončen        12
rodičovská      6
Name: count, dtype: int64

utvar
Laboratoře             31
Správa budov           27
IT oddělení            22
Knihovna               21
Studijní oddělení      20
Ekonomické oddělení    19
Personální oddělení     8
Marketing               6
                        5
Vedení                  5
Laboratore              3
Studijní                1
Studijni oddeleni       1
Sprava budov            1
SPRÁVA BUDOV            1
správa budov            1
Personalni oddeleni     1
VEDENÍ                  1
Name: count, dtype: int64

pozice
Laborant          28
Specialista       24
Referent          24
Technik           20
Technik údržby    16
Knihovník         15
Vedoucí           13
Účetní            12
Asistent          10
Řidič              8
Vývojář            4
Name: count, dtype: int64



Detecting which date formats appear in the employee dates, and whether more than one format (or an invalid value) is mixed in one column.

In [13]:
for column in ["datum_nastupu", "datum_ukonceni"]:
    print(column)
    print(value_shapes(employees_df[column]))

datum_nastupu
datum_nastupu
9999-99-99     169
9. 9. 9999       3
99. 9. 9999      2
Name: count, dtype: int64
datum_ukonceni
datum_ukonceni
              158
9999-99-99     16
Name: count, dtype: int64


Detecting duplicate employee ids or emails, and whether every id follows one consistent pattern.

In [14]:
print(f"duplicate id_zamestnance: {employees_df['id_zamestnance'].duplicated().sum()}")
print(f"duplicate email: {employees_df['email'].str.lower().duplicated().sum()}")

duplicate id_zamestnance: 0
duplicate email: 2


Checking whether the id of employees follows one consistent pattern

In [15]:
print("id_zamestnance patterns:")
print(value_shapes(employees_df["id_zamestnance"]))

id_zamestnance patterns:
id_zamestnance
A9999    174
Name: count, dtype: int64


Detecting the same person under two ids: same first name, department, position, start date and manager.

In [16]:
same_person_columns = ["jmeno", "utvar", "pozice", "datum_nastupu", "vedouci_email"]
shown_columns = ["id_zamestnance", "prijmeni", "jmeno", "utvar", "pozice", "datum_nastupu"]
is_same_person = employees_df.duplicated(same_person_columns, keep=False)
same_person_df = employees_df.loc[is_same_person, shown_columns]
same_person_df.sort_values(shown_columns[2:])

,id_zamestnance,prijmeni,jmeno,utvar,pozice,datum_nastupu
154,E0155,Novotný,Adam,Personální oddělení,Referent,2018-04-21
170,E1000,Novotny,Adam,Personální oddělení,Referent,2018-04-21
41,E0042,Zdražil,Jaroslav,Ekonomické oddělení,Účetní,2016-12-26
171,E1001,Zdražiy,Jaroslav,Ekonomické oddělení,Účetní,2016-12-26
69,E0070,Hruška,Petr,Knihovna,Knihovník,2022-08-07
173,E1003,Hrušky,Petr,Knihovna,Knihovník,2022-08-07
123,E0124,Havlíková,Petra,Laboratoře,Laborant,2015-09-01
172,E1002,Havlikova,Petra,Laboratoře,Laborant,2015-09-01


Detecting malformed emails: embedded spaces, a comma instead of a dot, or a missing domain.

In [17]:
is_malformed_email = ~employees_df["email"].str.fullmatch(r"[^@\s,]+@[^@\s,]+\.[a-z]+")
print(f"malformed email: {is_malformed_email.sum()}")
employees_df.loc[is_malformed_email, "id_zamestnance"].tolist()

malformed email: 6


['E0002', 'E0099', 'E0121', 'E0129', 'E0150', 'E0157']

Detecting leading or trailing spaces in names, which break exact name comparisons.

In [18]:
for column in ["jmeno", "prijmeni"]:
    has_outer_spaces = employees_df[column] != employees_df[column].str.strip()
    print(f"{column} with outer spaces: {has_outer_spaces.sum()}")

jmeno with outer spaces: 8
prijmeni with outer spaces: 0


Detecting end dates that disagree with status: active people that still carry an end date, or terminated people without one.

In [19]:
pd.crosstab(employees_df["stav"], employees_df["datum_ukonceni"] != "")

datum_ukonceni,False,True
stav,,
aktivní,152,4
rodičovská,6,0
ukončen,0,12


Detecting manager references that point to no known employee: orphaned or placeholder `vedouci_email` values.

In [20]:
known_emails = set(employees_df["email"].str.lower())
unknown_managers_df = employees_df[
    (employees_df["vedouci_email"] != "")
    & ~employees_df["vedouci_email"].str.lower().isin(known_emails)
]
print(f"vedouci_email not found among employees: {len(unknown_managers_df)}")
unknown_managers_df[["id_zamestnance", "vedouci_email"]].head(20)

vedouci_email not found among employees: 3


,id_zamestnance,vedouci_email
22,E0023,nezname.jmeno@vzorova.example
79,E0080,nezname.jmeno@vzorova.example
96,E0097,nezname.jmeno@vzorova.example


## 4. Completed trainings (`absolvovana_skoleni.csv`)

In [21]:
completed_df.head(10)

,id_zaznamu,id_zamestnance,zamestnanec,kod_skoleni,datum_absolvovani,platnost_do,poskytovatel,cena_kc,poznamka
0,Z000001,E0131,Vacková Hana,BOZP-Z,1900-01-01,1902-01-01,Interní školitel,"0,00",
1,Z000002,E0149,Navrátil Miroslav,BOZP-Z,1. 1. 1900,1. 1. 1902,Interní školitel,"0,00",
2,Z000003,E0024,Blažek Adam,BOZP-Z,2019-01-06,2021-01-06,Interní školitel,"450,00",
3,Z000004,E0063,Nováková Klára,GDPR,6. 1. 2019,6. 1. 2022,Vzorový školicí ústav,"700,00",
4,Z000005,E0088,Matoušková Adéla,POZ-OCH,6. 1. 2019,6. 1. 2021,E-learning portál,"380,00",online verze
5,Z000006,E0096,Novák Daniel,POZ-OCH,6. 1. 2019,6. 1. 2021,Vzorový školicí ústav,"0,00",
6,Z000007,E0118,Konečný Pavel,BOZP-Z,6. 1. 2019,6. 1. 2021,Vzorový školicí ústav,"550,00",
7,Z000008,E0067,Poláková Kateřina,BOZP-Z,7. 1. 2019,7. 1. 2021,Externí lektor,"550,00",absolvováno v zahraničí
8,Z000009,E0094,Pospíšil Vladimír,GDPR,07.01.2019,07.01.2024,Vzorový školicí ústav,"550,00",
9,Z000010,E0103,Prokopová Lenka,GDPR,7. 1. 2019,7. 1. 2022,E-learning portál,"650,00",


In [22]:
profile_columns(completed_df)

,empty,unique,example
id_zaznamu,0,2853,Z000001
id_zamestnance,2,182,E0131
zamestnanec,2,174,Vacková Hana
kod_skoleni,2,31,BOZP-Z
datum_absolvovani,2,1985,1900-01-01
platnost_do,49,2057,1902-01-01
poskytovatel,2,10,Interní školitel
cena_kc,21,47,"0,00"
poznamka,2599,14,online verze


Detecting the mix of formats in the completion and validity dates (ISO, `d. m. Y`, and any values that are not dates at all).

In [23]:
for column in ["datum_absolvovani", "platnost_do"]:
    print(column)
    print(value_shapes(completed_df[column]))
    print()

datum_absolvovani
datum_absolvovani
99. 9. 9999          968
9999-99-99           705
99.99.9999           462
9. 9. 9999           397
99. 99. 9999         242
9. 99. 9999           88
99999                  3
                       2
aaaaa_aaaaaaaaaaa      1
Name: count, dtype: int64

platnost_do
platnost_do
99. 9. 9999     954
9999-99-99      692
99.99.9999      449
9. 9. 9999      390
99. 99. 9999    243
9. 99. 9999      87
                 49
99999             3
aaaaaaaa_aa       1
Name: count, dtype: int64



Detecting training codes that match no catalog code: typos, casing or spacing variants, and genuinely unknown codes.

In [24]:
print("kod_skoleni not in catalog:")
completed_df.loc[~completed_df["kod_skoleni"].isin(catalog_df["kod"]), "kod_skoleni"].value_counts()

kod_skoleni not in catalog:


kod_skoleni
KYBER          6
BOZP-X         4
 BOZP-Z        2
BOZP-Z         2
POZ_OCH        2
               2
 POZ-OCH       2
POZ-OCH        2
POZ OCH        2
lead           1
BOZP Z         1
 KYBER         1
bozp-z         1
 GDPR          1
GDPR           1
kod_skoleni    1
GDPR-2         1
kyber          1
BOZP_Z         1
LAB-BEZ        1
KYB3R          1
Name: count, dtype: int64

Most codes above are catalog codes written differently: outer spaces (`'KYBER '` shows as `KYBER`), lowercase, or `_` and spaces instead of `-`. Detecting what is left after stripping, upper-casing and unifying separators. The empty value and `KOD-SKOLENI` come from junk rows (empty rows, repeated header).

In [25]:
normalized_codes = (
    completed_df["kod_skoleni"].str.strip().str.upper().str.replace(r"[_ ]+", "-", regex=True)
)
is_still_unknown = ~normalized_codes.isin(catalog_df["kod"])
print("kod_skoleni not in catalog after normalising:")
normalized_codes[is_still_unknown].value_counts()

kod_skoleni not in catalog after normalising:


kod_skoleni
BOZP-X         4
               2
KOD-SKOLENI    1
GDPR-2         1
KYB3R          1
Name: count, dtype: int64

Detecting completions linked to an employee id that is absent from the employee table.

In [26]:
print("id_zamestnance not in employees:")
is_unknown_employee = ~completed_df["id_zamestnance"].isin(employees_df["id_zamestnance"])
completed_df.loc[is_unknown_employee, "id_zamestnance"].value_counts()

id_zamestnance not in employees:


id_zamestnance
                  2
E9006             1
E9001             1
id_zamestnance    1
E9008             1
E9009             1
E9007             1
E9004             1
E9002             1
E9003             1
E9005             1
Name: count, dtype: int64

In [27]:
print(completed_df["kod_skoleni"].value_counts())

kod_skoleni
KYBER          838
BOZP-Z         573
POZ-OCH        528
GDPR           436
PRV-POM        212
LAB-BEZ        114
RIDIC           51
ZDVIH           46
ANGL            24
LEAD            10
KYBER            6
BOZP-X           4
 BOZP-Z          2
BOZP-Z           2
POZ_OCH          2
                 2
 POZ-OCH         2
POZ-OCH          2
POZ OCH          2
lead             1
BOZP Z           1
 KYBER           1
bozp-z           1
 GDPR            1
GDPR             1
kod_skoleni      1
GDPR-2           1
kyber            1
BOZP_Z           1
LAB-BEZ          1
KYB3R            1
Name: count, dtype: int64


In [28]:
print(completed_df["poskytovatel"].value_counts())


poskytovatel
E-learning portál        721
Vzorový školicí ústav    720
Externí lektor           713
Interní školitel         701
VZOROVÝ ŠKOLICÍ ÚSTAV      3
E-LEARNING PORTÁL          3
                           2
INTERNÍ ŠKOLITEL           2
EXTERNÍ LEKTOR             2
poskytovatel               1
Name: count, dtype: int64


Detecting the number format used for prices (decimal comma, thousands separators) so the cleaning step can parse it.

In [29]:
value_shapes(completed_df["cena_kc"])

cena_kc
999,99      1685
9,99         844
9 999,99     303
              21
999,- Aa       4
aaaaaa         4
-999,99        3
9,- Aa         2
aaaa_aa        1
9999           1
Name: count, dtype: int64

Detecting duplicate records: the same id written twice, and copies under another id. Dates are parsed and codes normalised first, so a copy in another date format is still caught.

In [30]:
valid_records_df = completed_df[completed_df["id_zaznamu"].str.fullmatch(r"Z\d{6}")]
print(f"repeated id_zaznamu: {valid_records_df['id_zaznamu'].duplicated().sum()}")

unique_records_df = valid_records_df.drop_duplicates("id_zaznamu").copy()
unique_records_df["completed_on"] = parse_mixed_dates(unique_records_df["datum_absolvovani"])
unique_records_df["code"] = (
    unique_records_df["kod_skoleni"].str.strip().str.upper().str.replace(r"[_ ]+", "-", regex=True)
)
key_columns = ["id_zamestnance", "code", "completed_on"]
is_copy = unique_records_df.duplicated(key_columns, keep=False)
print(f"copies under another id: {is_copy.sum()}")
unique_records_df.loc[
    is_copy, ["id_zaznamu", *key_columns, "poskytovatel"]
].sort_values(key_columns)

repeated id_zaznamu: 15


copies under another id: 20


,id_zaznamu,id_zamestnance,code,completed_on,poskytovatel
899,Z000895,E0017,BOZP-Z,2021-05-31,E-learning portál
2865,Z988277,E0017,BOZP-Z,2021-05-31,E-LEARNING PORTÁL
2582,Z002567,E0019,POZ-OCH,2025-08-29,Interní školitel
2860,Z951895,E0019,POZ-OCH,2025-08-29,INTERNÍ ŠKOLITEL
1469,Z001462,E0028,BOZP-Z,2023-03-20,Vzorový školicí ústav
2857,Z915067,E0028,BOZP-Z,2023-03-20,VZOROVÝ ŠKOLICÍ ÚSTAV
1097,Z001090,E0060,POZ-OCH,2022-02-27,Vzorový školicí ústav
2862,Z982602,E0060,POZ-OCH,2022-02-27,VZOROVÝ ŠKOLICÍ ÚSTAV
1545,Z001538,E0062,BOZP-Z,2023-04-25,E-learning portál
2861,Z960898,E0062,BOZP-Z,2023-04-25,E-LEARNING PORTÁL


Splitting the name mismatches by cause: extra spaces, letter case, dropped diacritics, and what is left as a real mismatch.

In [31]:
full_names = employees_df["prijmeni"] + " " + employees_df["jmeno"]
full_name_by_id = full_names.set_axis(employees_df["id_zamestnance"])
named_df = unique_records_df.copy()
named_df["expected"] = named_df["id_zamestnance"].map(full_name_by_id)
has_known_employee = named_df["expected"].notna()
is_name_different = named_df["zamestnanec"] != named_df["expected"]
mismatch_df = named_df[has_known_employee & is_name_different]
print(f"name differs from the employee record: {len(mismatch_df)}")

name differs from the employee record: 121


In [32]:
actual = mismatch_df["zamestnanec"].str.split().str.join(" ")
expected = mismatch_df["expected"].str.split().str.join(" ")
actual_lower = actual.str.lower()
expected_lower = expected.str.lower()
actual_plain = actual_lower.map(remove_diacritics)
expected_plain = expected_lower.map(remove_diacritics)

equal_ignoring_spaces = actual == expected
equal_ignoring_case = actual_lower == expected_lower
equal_ignoring_diacritics = actual_plain == expected_plain

print(f"extra spaces only: {equal_ignoring_spaces.sum()}")
print(f"letter case only: {(equal_ignoring_case & ~equal_ignoring_spaces).sum()}")
print(f"diacritics only: {(equal_ignoring_diacritics & ~equal_ignoring_case).sum()}")
print(f"real mismatch: {(~equal_ignoring_diacritics).sum()}")
mismatch_df.loc[~equal_ignoring_diacritics, ["id_zaznamu", "id_zamestnance"]]

extra spaces only: 21
letter case only: 95
diacritics only: 0
real mismatch: 5


,id_zaznamu,id_zamestnance
564,Z000561,E0029
595,Z000592,E0143
842,Z000838,E0094
1378,Z001371,E0068
1571,Z001564,E0052


Detecting completions dated before the employee was hired, such as the `1900-01-01` placeholder.

In [33]:
hire_dates = parse_mixed_dates(employees_df.set_index("id_zamestnance")["datum_nastupu"])
employee_hired_on = unique_records_df["id_zamestnance"].map(hire_dates)
is_before_hire = unique_records_df["completed_on"] < employee_hired_on
print(f"completed before the employee was hired: {is_before_hire.sum()}")
unique_records_df.loc[is_before_hire, ["id_zaznamu", "id_zamestnance", "datum_absolvovani"]]

completed before the employee was hired: 2


,id_zaznamu,id_zamestnance,datum_absolvovani
0,Z000001,E0131,1900-01-01
1,Z000002,E0149,1. 1. 1900


Parsing record dates and codes once for the next checks. Junk rows are skipped; the 3 Excel serial dates stay unparsed.

In [34]:
records_df = completed_df[completed_df["id_zaznamu"].str.fullmatch(r"Z\d{6}")].copy()
records_df["completed_on"] = parse_mixed_dates(records_df["datum_absolvovani"])
records_df["valid_until"] = parse_mixed_dates(records_df["platnost_do"])
records_df["code"] = (
    records_df["kod_skoleni"].str.strip().str.upper().str.replace(r"[_ ]+", "-", regex=True)
)

Detecting completions dated after the export date, which cannot be finished trainings.

In [35]:
is_future = records_df["completed_on"] > EXPORT_DATE
print(f"dated after export: {is_future.sum()}")
records_df.loc[is_future, ["id_zaznamu", "datum_absolvovani", "poznamka"]]

dated after export: 5


,id_zaznamu,datum_absolvovani,poznamka
2852,Z002836,27.10.2026,zadáno předem
2853,Z002837,25.11.2026,zadáno předem
2854,Z002838,28. 11. 2026,zadáno předem
2855,Z002839,4. 12. 2026,zadáno předem
2856,Z002840,2026-12-31,zadáno předem


Detecting `platnost_do` values that disagree with completion date + catalog validity.

In [36]:
validity_months = pd.to_numeric(
    catalog_df.set_index("kod")["platnost_mesicu"].replace({"2 roky": "24"}),
    errors="coerce",
)
records_df["catalog_months"] = records_df["code"].map(validity_months)
comparable_df = records_df[
    (records_df["catalog_months"] > 0)
    & records_df["completed_on"].notna()
    & records_df["valid_until"].notna()
].copy()
comparable_df["expected_until"] = [
    completed_on + pd.DateOffset(months=int(months))
    for completed_on, months in zip(comparable_df["completed_on"], comparable_df["catalog_months"])
]
is_different = comparable_df["valid_until"] != comparable_df["expected_until"]
print(f"comparable records: {len(comparable_df)}, platnost_do differs: {is_different.sum()}")
print(is_different.groupby(comparable_df["code"]).sum())

comparable records: 2809, platnost_do differs: 33


code
BOZP-Z     5
GDPR       8
KYBER      7
LAB-BEZ    0
POZ-OCH    8
PRV-POM    4
RIDIC      1
ZDVIH      0
dtype: int64


How far off the disagreeing `platnost_do` values are, in months.

In [37]:
DAYS_PER_MONTH = 30.44
days_off = (comparable_df["valid_until"] - comparable_df["expected_until"]).dt.days
(days_off[is_different] / DAYS_PER_MONTH).round().value_counts()

-6.0     11
 24.0     8
 12.0     4
-12.0     2
-15.0     2
-29.0     1
-30.0     1
-41.0     1
-28.0     1
-26.0     1
-14.0     1
Name: count, dtype: int64

Detecting empty `platnost_do` per training. ANGL and LEAD have no catalog validity, so they are left out.

In [38]:
is_empty_validity = records_df["platnost_do"].str.strip() == ""
has_catalog_validity = records_df["catalog_months"] > 0
records_df.loc[is_empty_validity & has_catalog_validity, "code"].value_counts()

code
POZ-OCH    4
BOZP-Z     3
KYBER      3
PRV-POM    2
Name: count, dtype: int64

## 5. Training dates (`terminy_skoleni.csv`)

Small enough to read in full.

In [39]:
dates_df

,id_terminu,kod_skoleni,datum,cas,misto,kapacita,obsazeno,cena_na_osobu,stav
0,T001,KYBER,2026-10-14,09:00-12:00,Aula A,25,22,800,plánováno
1,T002,KYBER,2026-10-28,09:00-12:00,Aula A,40,0,800,plánováno
2,T003,KYBER,2026-11-11,09:00-12:00,Aula A,30,5,800,plánováno
3,T004,KYBER,2026-11-25,09:00-12:00,Aula A,30,0,800,plánováno
4,T005,KYBER,2026-12-09,09:00-12:00,Aula A,25,0,800,plánováno
5,T006,KYBER,2026-12-16,13:00-15:00,Online,50,0,800,plánováno
6,T007,KYBER,2026-11-14,09:00-12:00,Online,60,0,800,plánováno
7,T008,KYBER,2026-09-23,09:00-12:00,Aula A,30,0,800,plánováno
8,T009,KYBER,2026-11-11,09:00-12:00,Aula A,30,5,800,plánováno
9,T010,KYB3R,2026-12-02,09:00-12:00,Aula B,20,0,800,plánováno


In [40]:
profile_columns(dates_df)

,empty,unique,example
id_terminu,0,34,T001
kod_skoleni,0,10,KYBER
datum,0,31,2026-10-14
cas,0,10,09:00-12:00
misto,0,9,Aula A
kapacita,0,11,25
obsazeno,0,13,22
cena_na_osobu,2,11,800
stav,0,2,plánováno


Detecting the status and location vocabularies, any term whose training code is not in the catalog, duplicate term ids, and the date format.

In [41]:
for column in ["stav", "misto", "cas"]:
    print(dates_df[column].value_counts())
    print()
print(f"kod_skoleni not in catalog: {(~dates_df['kod_skoleni'].isin(catalog_df['kod'])).sum()}")
print(f"duplicate id_terminu: {dates_df['id_terminu'].duplicated().sum()}")
print(value_shapes(dates_df["datum"]))

stav
plánováno    32
zrušeno       2
Name: count, dtype: int64

misto
Aula A                12
Online                 5
Učebna 1               4
Učebna 2               4
Autoškola Vzor         3
Laboratoř 3            2
Dvůr – sklad           2
Aula B                 1
Seminární místnost     1
Name: count, dtype: int64

cas
09:00-12:00    9
08:00-11:00    5
10:00-12:00    4
09:00-15:00    4
14:00-16:00    3
08:00-12:00    3
09:00-13:00    2
07:30-15:30    2
13:00-15:00    1
09:00-17:00    1
Name: count, dtype: int64

kod_skoleni not in catalog: 1
duplicate id_terminu: 0
datum
9999-99-99     33
99. 9. 9999     1
Name: count, dtype: int64


Detecting non-numeric values in the numeric columns, and any term booked beyond its capacity (`obsazeno > kapacita`).

In [42]:
number_columns = ["kapacita", "obsazeno", "cena_na_osobu"]
capacity_df = dates_df[number_columns].apply(pd.to_numeric, errors="coerce")
print(f"non-numeric values: {capacity_df.isna().sum().to_dict()}")
print("obsazeno > kapacita:")
dates_df[capacity_df["obsazeno"] > capacity_df["kapacita"]]

non-numeric values: {'kapacita': 0, 'obsazeno': 0, 'cena_na_osobu': 3}
obsazeno > kapacita:


,id_terminu,kod_skoleni,datum,cas,misto,kapacita,obsazeno,cena_na_osobu,stav
15,T016,BOZP-Z,2026-11-04,08:00-11:00,Aula A,30,31,450,plánováno
29,T030,LAB-BEZ,2026-10-29,09:00-13:00,Laboratoř 3,20,21,1500,plánováno


Detecting which terms fall before or after the export date, to separate usable future terms from ones already in the past.

In [43]:
parsed_dates = pd.to_datetime(dates_df["datum"], format="%Y-%m-%d", errors="coerce")
timeline_df = dates_df.assign(days_from_export=(parsed_dates - EXPORT_DATE).dt.days)
timeline_df = timeline_df[["id_terminu", "kod_skoleni", "datum", "stav", "days_from_export"]]
timeline_df.sort_values("days_from_export").head(15)

,id_terminu,kod_skoleni,datum,stav,days_from_export
18,T019,BOZP-Z,2026-09-16,plánováno,-15.0
7,T008,KYBER,2026-09-23,plánováno,-8.0
21,T022,GDPR,2026-09-30,plánováno,-1.0
14,T015,BOZP-Z,2026-10-07,plánováno,6.0
0,T001,KYBER,2026-10-14,plánováno,13.0
26,T027,RIDIC,2026-10-15,plánováno,14.0
10,T011,POZ-OCH,2026-10-20,plánováno,19.0
22,T023,PRV-POM,2026-10-22,plánováno,21.0
1,T002,KYBER,2026-10-28,plánováno,27.0
29,T030,LAB-BEZ,2026-10-29,plánováno,28.0


## 6. Insights

Insights and decisions are listed in `01_cleaning_ntb.ipynb`.